# Power System Load Type Prediction

Run `src/train.py` from the VS Code terminal for the full training pipeline. This notebook provides EDA and a chronological last-month split preview.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
files = list((ROOT / 'data').glob('*.xlsx')) + list((ROOT / 'data').glob('*.xls'))
if not files:
    raise FileNotFoundError('Place load_data.xlsx in the data/ folder.')
df = pd.read_excel(files[0])
print('File:', files[0].name)
print('Shape:', df.shape)
display(df.head())
print(df.info())

## Data quality

In [ ]:
display(df.isna().sum().sort_values(ascending=False).to_frame('missing_values'))
print('Duplicate rows:', df.duplicated().sum())
display(df.describe(include='all').T)

## Target distribution

In [ ]:
target_candidates = [c for c in df.columns if str(c).strip().lower().replace('_', ' ') == 'load type']
if target_candidates:
    target = target_candidates[0]
    ax = df[target].value_counts(dropna=False).plot(kind='bar', figsize=(7,4), title='Load Type Distribution')
    ax.set_xlabel('Load Type'); ax.set_ylabel('Count'); plt.tight_layout(); plt.show()
else:
    print('Target column not detected. Actual columns:', list(df.columns))

## Last-month test split preview

This uses the latest calendar month in `Date` as the test set, matching the problem statement.

In [ ]:
date_candidates = [c for c in df.columns if str(c).strip().lower() in ('date','datetime','timestamp')]
if not date_candidates:
    print('Date column not detected. Actual columns:', list(df.columns))
else:
    date_col = date_candidates[0]
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce', dayfirst=True)
    clean = df.dropna(subset=[date_col]).sort_values(date_col)
    last_month = clean[date_col].max().to_period('M')
    train = clean[clean[date_col].dt.to_period('M') != last_month]
    test = clean[clean[date_col].dt.to_period('M') == last_month]
    print('Latest test month:', last_month)
    print('Training rows:', len(train))
    print('Test rows:', len(test))